# 🌲 Phase 2: Anomaly Detection Using Isolation Forest
### **SecureBank AI — Hybrid Fraud Intelligence System**

#### **Module Objectives:**
1. **Unsupervised Anomaly Modeling**: Train an `IsolationForest` on normal transaction behavior to isolate outliers without requiring fraud labels during fitting.
2. **Novelty & Zero-Day Detection**: Complement supervised Random Forest (which detects known fraud patterns) with anomaly detection (which catches novel or unseen fraud vectors).
3. **Decision Function Scoring**: Extract continuous anomaly scores where more negative values indicate deeper isolation (higher abnormality).
4. **Evaluation & Comparison**: Benchmark unsupervised anomaly detection against ground-truth labels and compare against supervised classification.
5. **Artifact Export**: Serialize `models/isolation_forest.pkl` for immediate runtime integration into the hybrid risk scoring engine.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)
print("Environment initialized.")

## 1. Load Preprocessed Training and Testing Data
We load `data/processed/train_processed.csv` and `data/processed/test_processed.csv`.

In [ ]:
train_path = "../data/processed/train_processed.csv"
test_path = "../data/processed/test_processed.csv"
if not os.path.exists(train_path):
    train_path = "data/processed/train_processed.csv"
    test_path = "data/processed/test_processed.csv"

print(f"Loading data from {train_path}...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# In semi-supervised anomaly detection, we train on normal transactions so the model learns normal density
X_train_normal = train_df[train_df["Class"] == 0].drop(columns=["Class"])
# Subsample 60,000 normal instances for fast, robust density modeling
if len(X_train_normal) > 60000:
    X_train_iso = X_train_normal.sample(n=60000, random_state=42)
else:
    X_train_iso = X_train_normal

X_test = test_df.drop(columns=["Class"])
y_test = test_df["Class"]

print(f"Training instances for normal profile: {len(X_train_iso):,}")
print(f"Test instances: {len(X_test):,} (Fraud: {y_test.sum()}, Normal: {(y_test == 0).sum()})")

## 2. Train Isolation Forest Anomaly Detector
Isolation Forest isolates anomalies by randomly selecting a feature and randomly selecting a split value.
Because anomalies are few and structurally distinct, they require noticeably fewer random splits to isolate than normal points.

In [ ]:
start_time = time.perf_counter()

isolation_model = IsolationForest(
    n_estimators=150,
    max_samples=0.5,
    contamination=0.003,  # Expected empirical outlier threshold
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train_iso)
training_time = time.perf_counter() - start_time
print(f"Isolation Forest training completed in {training_time:.2f} seconds.")

## 3. Anomaly Prediction and Scoring
- In Scikit-Learn: `predict()` returns `-1` for anomalies and `1` for normal instances.
- `decision_function()` returns negative scores for outliers and positive scores for inliers.

In [ ]:
raw_predictions = isolation_model.predict(X_test)
# Map: -1 (anomaly) -> 1, 1 (normal) -> 0
y_anomaly_pred = (raw_predictions == -1).astype(int)

# Decision function: more negative = more anomalous
raw_scores = isolation_model.decision_function(X_test)
# Invert so that higher score = higher anomaly likelihood
anomaly_risk_score = -raw_scores

auc = roc_auc_score(y_test, anomaly_risk_score)
print(f"Isolation Forest ROC-AUC Score: {auc:.4f}")
print("\nClassification Report (Unsupervised):\n")
print(classification_report(y_test, y_anomaly_pred, target_names=['Normal', 'Anomaly/Fraud'], zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_anomaly_pred))

## 4. Anomaly Score Distribution Analysis
We inspect how the anomaly score separates genuine transactions from actual fraud cases.

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(anomaly_risk_score[y_test == 0], bins=60, alpha=0.6, color='#2563eb', density=True, label='Legitimate Transactions')
plt.hist(anomaly_risk_score[y_test == 1], bins=60, alpha=0.8, color='#ef4444', density=True, label='Fraud Transactions')
plt.axvline(np.percentile(anomaly_risk_score, 99.7), color='#f59e0b', linestyle='--', linewidth=2, label='99.7% Anomaly Threshold')

plt.title(f"Isolation Forest Anomaly Score Distribution (ROC-AUC: {auc:.3f})", fontsize=13, fontweight='bold')
plt.xlabel("Anomaly Severity Score (Higher = More Abnormal)", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/isolation_forest_distribution.png", dpi=300)
plt.show()

## 5. Save Isolation Forest Model Artifact
We save the fitted model to `models/isolation_forest.pkl` so the Streamlit dashboard and hybrid prediction engine can use it directly.

In [ ]:
model_path = "../models/isolation_forest.pkl"
if not os.path.exists("../models"):
    model_path = "models/isolation_forest.pkl"

joblib.dump(isolation_model, model_path)
print(f"Successfully saved Isolation Forest to: {model_path}")